In [1]:
import numpy as np
import dask , dask.distributed
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy
import cmocean.cm as cmo
import warnings
warnings.simplefilter('ignore')
import dask_jobqueue
from astropy.convolution import Box2DKernel, convolve
import pandas as pd

In this file, the data was merged into a single file for more practicability later

# monthly

In [ ]:
# windstress_1PctTo2X.nc
# windstress_ctrl.nc

In [ ]:
#SST, SSH, dic_stf, o2_stf, sens_heat, evap_heat, mld, jp_all =jp_uptake-remin-recycle
#tau_x, tau_y, wind_stress

In [ ]:
path_to='/gxfs_work/geomar/smomw577/mesoscale_eddies/MOM5_concat/0181-0190/'
path_from_ctrl='/gxfs_work/geomar/smomw577/mesoscale_eddies/MOM5/ctrl/'
path_from_1PctTo2X='/gxfs_work/geomar/smomw577/mesoscale_eddies/MOM5/1PctTo2X/'

In [ ]:
def merging(path):
    ice_daily=xr.open_mfdataset(path+'01*0101.ice_daily.nc')[['SSH', 'SST']]#yt, xt
    bio=xr.open_mfdataset(path+'01*0101.ocean_minibling_100m.nc')[['jp_uptake', 'jp_reminp', 'jp_recycle']] #yt_ocean, xt_ocean
    hf=xr.open_mfdataset(path+'01*0101.ocean_bdy_flux.nc')[['sens_heat', 'evap_heat']] #yt_ocean, xt_ocean
    gas_flux=xr.open_mfdataset(path+'01*0101.ocean_minibling_term_src.nc')[['o2_stf', 'dic_stf']] *60*60*24*365 #yt_ocean, xt_ocean
    mld=xr.open_mfdataset(path+'01*0101.ocean.nc')[['mld']] #yt_ocean, xt_ocean
    
    ice_daily=ice_daily.interp_like(bio.jp_uptake)
    ds_out = xr.merge([
    ice_daily,
    bio,
    hf,
    gas_flux,
    mld
    ])
    
    ds_out['jp_all'] = bio.jp_uptake - bio.jp_reminp - bio.jp_recycle
    
    ds_out = ds_out.drop_vars(['jp_uptake', 'jp_reminp', 'jp_recycle'])
    if 'ctrl' in path:
        ds_out.to_netcdf(path_to + 'MOM5_control_monthly_0181-0190_all.nc')
    elif '1PctTo2X' in path:
        ds_out.to_netcdf(path_to + 'MOM5_1PctTo2X_monthly_0181-0190_all.nc.nc')

In [ ]:
merging(path_from_ctrl)
merging(path_from_1PctTo2X)

## wind

files windstress_ctrl.nc and windstress_1PctTo2X.nc were created with cdo and contain only tau_x and tau_y